# Claude Opus 4.8 — Cookbook
Practical recipes, examples, and best practices for building with Claude Opus 4.8.
> Source: Anthropic announcement — https://www.anthropic.com/news/claude-opus-4-8

## 0) Setup and Installation
- **Prereqs:** Python 3.8+, `requests`, and an API key for the Claude platform.
- **Install (Python):** `pip install requests`

In [ ]:
# Run this once in a fresh environment
# %pip install -U requests

**Authentication**
Set your API key in the environment before running examples.

In [ ]:
import os
API_KEY = os.getenv("CLAUDE_API_KEY")
if not API_KEY:
    raise ValueError("Set CLAUDE_API_KEY before running this notebook.")
print("CLAUDE_API_KEY found (hidden)")

## 1) Basic Usage (HTTP / curl / requests)
The Messages API accepts `system` entries inside the `messages` array. Replace the endpoint with the exact URL from Platform docs.

**curl example (replace endpoint and key):**

In [ ]:
# curl example (display only)
curl_payload = '''
POST /v1/messages HTTP/1.1
Host: api.claude.example
Authorization: Bearer $CLAUDE_API_KEY
Content-Type: application/json

{
  "model": "claude-opus-4-8",
  "messages": [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "List 5 checks to harden a Python API before production."}
  ],
  "effort": "high"
}
'''
print(curl_payload)

**Python requests example:**

In [ ]:
import requests, json, os
API_KEY = os.getenv("CLAUDE_API_KEY")
ENDPOINT = os.getenv("CLAUDE_API_ENDPOINT", "https://api.claude.example/v1/messages")  # replace with real endpoint
payload = {
    "model": "claude-opus-4-8",
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "List 5 checks to harden a Python API before production."}
    ],
    "effort": "high",
}
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}
# Warning: network call commented out for safety in notebooks. Uncomment to run.
# resp = requests.post(ENDPOINT, headers=headers, json=payload, timeout=60)
# print(resp.json())
print("Prepared request to", ENDPOINT)

## 2) Framework Integration
Many frameworks expect a simple call-based LLM interface. Wrap HTTP calls in a minimal class to adapt to your framework.

In [ ]:
class ClaudeClient:
    def __init__(self, endpoint, api_key):
        self.endpoint = endpoint
        self.api_key = api_key

    def send_messages(self, messages, model="claude-opus-4-8", effort="high"):
        import requests
        payload = {
            "model": model,
            "messages": messages,
            "effort": effort,
        }
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        # network call commented: replace with real endpoint when ready
        # r = requests.post(self.endpoint, headers=headers, json=payload, timeout=60)
        # return r.json()
        return {"mock": "response"}

## 3) Building Agents (tool-calling pattern)
Below is a minimal local pattern: the assistant may request a tool; the client executes it and returns the result as a `tool` message. Adapt the parsing to the provider response format.

In [ ]:
import json

def check_service(name):
    mapping = {
        "postgres": "ok",
        "redis": "ok",
        "legacy-cache": "deprecated",
    }
    return {"dependency": name, "status": mapping.get(name, "unknown")}

# Simulated assistant asking for a tool call (mocked)
assistant_call = {
    "type": "tool_call",
    "tool": "check_service",
    "args": {"name": "legacy-cache"}
}
if assistant_call["type"] == "tool_call":
    if assistant_call["tool"] == "check_service":
        result = check_service(assistant_call["args"]["name"])
        print(json.dumps(result))

## 4) Advanced Applications
Patterns for long-context QA, retrieval, and structured outputs.

In [ ]:
def chunk_text(text: str, chunk_size: int = 3000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Example: assemble a context from document chunks
long_doc = "Replace with your long document content." * 200
chunks = chunk_text(long_doc)
context = "\n\n".join(chunks[:20])
print("Prepared context length:", len(context))

## Structured JSON outputs
Ask the model to `Return valid JSON only.` and post-validate with `json.loads`.

In [ ]:
# Example: validate JSON-only response (mock)
mock_resp = '[{
:
,
:
,
:
}]'
import json
try:
    parsed = json.loads(mock_resp)
    print("Parsed JSON with", len(parsed), "items")
except Exception as e:
    print("Invalid JSON response:", e)

## Usage & Token tracking
If the provider returns a `usage` object, print token counts for cost tracking. Wrap access in `try/except` as fields may be absent.

In [ ]:
# Example: safe access to usage fields (mock)
resp = {"usage": {"prompt_tokens": 10, "completion_tokens": 90, "total_tokens": 100}}
usage = resp.get("usage", {})
print("prompt_tokens:", usage.get("prompt_tokens"))
print("completion_tokens:", usage.get("completion_tokens"))
print("total_tokens:", usage.get("total_tokens"))

## Model-Specific Notes & Safety
- Default `effort` is `high`. Use `extra` (xhigh) or `max` for more intensive reasoning.
- Opus 4.8 improves honesty and tool usage, but add verification for high-stakes outputs.

## References
- Announcement: https://www.anthropic.com/news/claude-opus-4-8
- System card: https://www.anthropic.com/claude-opus-4-8-system-card
- Platform docs: https://platform.claude.com/docs
---
Cookbook skeleton complete. Replace placeholders and uncomment network calls after setting `CLAUDE_API_KEY` and the real `CLAUDE_API_ENDPOINT`.